# M15: RAG test for the Spending Analyzer

**Question from the handout:** does retrieving evidence improve our answers, or only make them *look* more credible?

**The one job we test (our "bounded function"):** a new charge shows up on the statement. The app suggests a category **and points to the evidence it used** — past charges the student already labeled, or a line from our category guide. If there isn't good evidence, it should say so and ask the student instead of guessing.

This is the follow-up our A2 dossier asked for: *"can a student's past answers settle mixed stores like Target?"*

---

### RAG in plain words

RAG = **R**etrieval-**A**ugmented **G**eneration. Think of it as an open-book exam:

1. **Retrieval** — before answering, look things up in "the book" (our sources) and pull out the few passages that look most related to the question.
2. **Augmented** — paste those passages next to the question.
3. **Generation** — write the answer *using only those passages*, and say which passage each claim came from (the citation).

The whole pipeline in this notebook:

```
new charge ──► clean the text ──► find the top-k most similar chunks ──► answer step ──► category + chunk IDs cited
                                    (our "book" = 2 sources)             (or "ask the student")
```

### What is real and what is simulated

- **Real:** the data, the chunks, and the retrieval (actual similarity search, same method as A2).
- **Simulated:** the "generation" step. Instead of calling an LLM, we use a short, transparent rule that reads the retrieved chunks the way a careful LLM *should* (Step 4 explains it). We did this so every answer is repeatable and every citation can be checked automatically. Section 9 prints the exact prompt a real LLM would get, so you can paste it into Claude and compare.

**How to run:** in Colab, upload `transactions.csv` when asked (or keep the `data` folder next to this notebook). Then Runtime → Run all. Takes a few seconds.

In [6]:
import os
import re

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)

# Find the data: data/transactions.csv, or transactions.csv next to the notebook, or upload it in Colab
DATA = next((p for p in ["data/transactions.csv", "transactions.csv"] if os.path.exists(p)), None)
if DATA is None:
    try:
        from google.colab import files  # only exists on Colab
        print("Please upload transactions.csv")
        files.upload()
        DATA = "transactions.csv"
    except ImportError:
        raise FileNotFoundError("Put transactions.csv in a 'data' folder next to this notebook.")

df = pd.read_csv(DATA)
print(len(df), "charges from", df["date"].min(), "to", df["date"].max())

ModuleNotFoundError: No module named 'sklearn'

## Step 1. Clean up the bank text

Same cleanup as A2. Banks wrap the store name in junk (`POS PURCHASE`, `CHECKCARD 0904`, card numbers, store numbers). We strip it so the same store looks the same every time.

One side effect to remember for later: the cleanup **deletes any word containing a digit** (to remove store numbers). That will matter in case 5.

In [ ]:
PREFIXES = [r"^POS PURCHASE ", r"^CHECKCARD \d{4} ", r"^PURCHASE AUTHORIZED ON \d\d/\d\d "]


def normalize(desc: str) -> str:
    s = desc.upper()
    for p in PREFIXES:
        s = re.sub(p, "", s)
    s = re.sub(r" CARD \d{4}$", "", s)
    s = re.sub(r"(PPD|WEB) ID: ?\d+", "", s)
    s = re.sub(r"REF #\S+", "", s)
    s = s.replace("*", " * ")
    tokens = [t for t in s.split() if not re.search(r"\d", t)]  # drop any word with a digit in it
    return re.sub(r"\s+", " ", " ".join(tokens)).strip()


df["clean"] = df["description"].map(normalize)
df[["description", "clean"]].head(5)

,description,clean
0,ACH DEBIT RIVERVIEW PROP MGMT RENT 0726,ACH DEBIT RIVERVIEW PROP MGMT RENT
1,PATREON* MEMBERSHIP INTERNET CA,PATREON * MEMBERSHIP INTERNET CA
2,POS PURCHASE TST* LITTLE MOMO TROY NY,TST * LITTLE MOMO TROY NY
3,CHECKCARD 0703 LYFT *RIDE FRI 11PM LYFT.COM CA 65565592430,LYFT * RIDE FRI LYFT.COM CA
4,DIRECT DEP HUDSON DINING SVC PAYROLL PPD ID: 9145667201,DIRECT DEP HUDSON DINING SVC PAYROLL


## Step 2. Build the "book": two sources, cut into chunks

A **chunk** is one small piece of text that retrieval can return, with a permanent **ID** so we can cite it.

| Source | What it is | Chunk = | ID looks like |
|---|---|---|---|
| **S1 · Charge history** | The student's own labeled charges from **July and August** (143 charges), with the category and the note of what was bought | one charge | `H-049` (= charge #49) |
| **S2 · Category guide** | Our team's keyword list from A2 (e.g. `URGENT CARE` → Health) | one keyword | `G-HEALTH-04` |

**Why only July and August?** September (68 charges) is our test set. We pretend those charges are brand new and see if the history can answer them. If September were in the book, the system could just look up the answer (that's cheating, like having the answer key in the open book).

In [ ]:
hist = df[df["date"] < "2026-09-01"].copy()   # S1: the book
test = df[df["date"] >= "2026-09-01"].copy()  # September: the "new" charges we test on

s1 = pd.DataFrame({
    "chunk_id": [f"H-{i:03d}" for i in hist["id"]],
    "source": "S1 history",
    "text": hist["clean"].values,
    "date": hist["date"].values,
    "amount": hist["amount"].values,
    "category": hist["category"].values,
    "note": hist["note"].fillna("").values,
})

# S2: the keyword list we wrote for A2, one chunk per keyword
GUIDE = [
    ("Income", "INCOME", ["PAYROLL", "DIRECT DEP", "DIR DEP"]),
    ("Transfers", "TRANSF", ["VENMO", "ZELLE", "TRANSFER", "MOBILE PMT", "PAYMENT THANK YOU", "CASHOUT"]),
    ("Fees", "FEES", ["SERVICE FEE", "OVERDRAFT", "ATM FEE", "LATE FEE", "LATE PAYMENT", "VIOLATION", "FEE"]),
    ("Rent & Utilities", "RENT", ["RENT", "NATIONAL GRID", "SPECTRUM", "VERIZON", "T-MOBILE", "MINT MOBILE",
                                  "ELECTRIC", "UTILIT", "COMCAST", "XFINITY"]),
    ("Subscriptions", "SUBS", ["NETFLIX", "SPOTIFY", "HULU", "DISNEY", "APPLE.COM/BILL", "CHATGPT", "OPENAI",
                               "AMAZON PRIME", "PATREON", "YOUTUBE"]),
    ("Dining", "DINING", ["STARBUCKS", "DUNKIN", "CHIPOTLE", "MCDONALD", "DOORDASH", "GRUBHUB", "UBER * EATS",
                          "UBER EATS", "PANERA", "CAFE", "COFFEE", "PIZZA", "BAGEL", "BBQ", "BAR-B-Q", "GRILL",
                          "RESTAURANT"]),
    ("Groceries", "GROC", ["PRICE CHOPPER", "HANNAFORD", "ALDI", "TRADER JOE", "WHOLE FOODS", "WHOLEFDS",
                           "WEGMANS", "INSTACART", "COSTCO", "GROCERY", "MARKET", "SUPERMARKET", "FOOD"]),
    ("Transport", "TRANSP", ["UBER", "LYFT", "CDTA", "SHELL", "MOBIL", "SUNOCO", "EXXON", "STEWARTS", "GAS",
                             "FUEL", "PARKING", "AMTRAK", "MTA", "TRANSIT"]),
    ("Shopping", "SHOP", ["AMAZON", "AMZN", "TARGET", "WALMART", "WAL-MART", "WM SUPERCENTER", "BEST BUY", "ETSY",
                          "EBAY", "NIKE", "UNIQLO", "BOOKSTORE", "SHEIN"]),
    ("Health", "HEALTH", ["CVS", "WALGREENS", "PHARMACY", "FITNESS", "URGENT CARE", "MEDICAL", "DENTAL", "COPAY"]),
    ("Entertainment", "ENT", ["AMC", "REGAL", "STEAM", "TICKETMASTER", "PLAYSTATION", "XBOX", "NINTENDO", "BOWL",
                              "CINEMA", "THEATRE", "THEATER"]),
]
s2 = pd.DataFrame([
    {"chunk_id": f"G-{short}-{j:02d}", "source": "S2 guide", "text": kw, "date": "", "amount": np.nan,
     "category": cat, "note": "category guide keyword"}
    for cat, short, kws in GUIDE for j, kw in enumerate(kws)
])

chunks = pd.concat([s1, s2], ignore_index=True)
print(f"S1 history: {len(s1)} chunks   S2 guide: {len(s2)} chunks   total: {len(chunks)}")
chunks[chunks["chunk_id"].isin(["H-011", "H-049", "H-075", "H-126", "G-SHOP-02"])]

S1 history: 143 chunks   S2 guide: 112 chunks   total: 255


,chunk_id,source,text,date,amount,category,note
10,H-011,S1 history,TARGET LATHAM NY,2026-07-06,-20.27,Groceries,food run
48,H-049,S1 history,TARGET LATHAM NY,2026-07-21,-63.13,Shopping,apartment / dorm stuff
74,H-075,S1 history,TARGET LATHAM NY,2026-08-04,-22.51,Groceries,food run
125,H-126,S1 history,TARGET LATHAM NY,2026-08-25,-63.40,Shopping,apartment / dorm stuff
225,G-SHOP-02,S2 guide,TARGET,,NaN,Shopping,category guide keyword


Above: four Target charges from the history plus one guide chunk. Notice Target is a **mixed store**: $20 and $22 trips were *food runs* (Groceries), $63 trips were *dorm stuff* (Shopping). The store name alone can't tell them apart — that's the problem RAG is supposed to help with.

## Step 3. Retrieval: find the most similar chunks

The computer can't compare text directly, so we turn every chunk into a list of numbers (a **vector**):

- **TF-IDF on character pieces.** We chop text into overlapping 3–5 letter pieces (`TAR`, `ARG`, `RGE`, `TARG`, ...). Each piece gets a number: higher if it appears in this text, lower if it appears everywhere (pieces like ` NY` are in almost every charge, so they count for little). Same method as A2's similarity search.
- **Cosine similarity** compares two vectors and gives a score from **0** (nothing in common) to **1** (identical after cleaning).
- **Top-k** = keep the k highest-scoring chunks. **k is our A/B variable.**

Important: this is *lexical* similarity — it matches letters, not meaning. It has no idea that "urgent care" is medical. It only knows the letters match the guide keyword `URGENT CARE`.

When several chunks tie (e.g. all four Target chunks score 1.00), the one stored first (oldest) comes first. That's an arbitrary choice — keep it in mind.

In [ ]:
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)
chunk_vectors = vectorizer.fit_transform(chunks["text"])


def retrieve(description: str, k: int) -> pd.DataFrame:
    # Return the k chunks most similar to a (raw) bank description, best first
    q = vectorizer.transform([normalize(description)])
    scores = (q @ chunk_vectors.T).toarray()[0]
    order = np.argsort(-scores, kind="stable")[:k]  # 'stable' = ties keep their stored order
    out = chunks.iloc[order].copy()
    out.insert(1, "similarity", scores[order].round(2))
    return out


# Try it on one new charge from September
retrieve("POS PURCHASE TARGET T-1849 LATHAM NY", k=5)[["chunk_id", "similarity", "text", "amount", "category", "note"]]

,chunk_id,similarity,text,amount,category,note
10,H-011,1.00,TARGET LATHAM NY,-20.27,Groceries,food run
48,H-049,1.00,TARGET LATHAM NY,-63.13,Shopping,apartment / dorm stuff
74,H-075,1.00,TARGET LATHAM NY,-22.51,Groceries,food run
125,H-126,1.00,TARGET LATHAM NY,-63.40,Shopping,apartment / dorm stuff
225,G-SHOP-02,0.74,TARGET,NaN,Shopping,category guide keyword


## Step 4. The answer step (our stand-in for the LLM)

This is the "generation" part. A real RAG system would hand the retrieved chunks to an LLM with instructions like *"answer only from these chunks, cite them, say if they're not enough."* We wrote those instructions as a rule so we can check them exactly:

1. **Ignore weak matches.** Only chunks with similarity **≥ 0.50** count as evidence. If nothing is left → **"not enough evidence, ask the student."**
2. **If all the evidence agrees** on one category → answer it and cite those chunks.
3. **If the evidence disagrees** (a mixed store), keep only the *closest* text matches. If those still disagree, look at the **amount**: pick the past charge whose amount is closest to the new one.
4. **If no past charge is within 50% of the new amount**, or there's still a tie → **ask the student.**

With **k = 1**, rules 3 and 4 can never happen (there's only one chunk), so the answer is just "copy the category of the single closest chunk."

In [ ]:
MIN_SIM = 0.50     # rule 1
AMOUNT_TOL = 0.50  # rule 4: closest past amount must be within 50% of the new amount


def answer(retrieved: pd.DataFrame, amount: float):
    # Return (category or 'ASK STUDENT', cited chunk IDs, one-sentence answer)
    ev = retrieved[retrieved["similarity"] >= MIN_SIM]
    if ev.empty:
        return "ASK STUDENT", [], "Not enough evidence in the sources. Please ask the student."
    if ev["category"].nunique() == 1:
        cat = ev["category"].iloc[0]
        return cat, list(ev["chunk_id"]), f"{cat}: matching past charges / guide entries say {cat}."
    top = ev[ev["similarity"] >= ev["similarity"].max() - 1e-9]  # closest text matches only
    if top["category"].nunique() == 1:
        cat = top["category"].iloc[0]
        return cat, list(top["chunk_id"]), f"{cat}: the closest matches all say {cat}."
    priced = top.dropna(subset=["amount"])
    gap = (priced["amount"].abs() - abs(amount)).abs()
    if priced.empty or gap.min() > AMOUNT_TOL * abs(amount):
        return "ASK STUDENT", [], "Mixed store and no past charge of a similar amount. Please ask the student."
    best = priced[gap <= gap.min() + 1e-9]
    if best["category"].nunique() > 1:
        return "ASK STUDENT", [], "Mixed store and the evidence is tied. Please ask the student."
    cat, row = best["category"].iloc[0], best.iloc[0]
    return cat, list(best["chunk_id"]), (f"{cat}: mixed store; the past charge closest in amount "
                                         f"(${abs(row['amount']):.2f}, note: '{row['note']}') was {cat}.")


def build_prompt(description: str, amount: float, retrieved: pd.DataFrame) -> str:
    # The prompt a real LLM would get. We only print it (section 9); the rule above stands in for the LLM.
    lines = [f"[{r.chunk_id}] {r.text} | {'' if pd.isna(r.amount) else f'${abs(r.amount):.2f} | '}"
             f"category: {r.category}{' | note: ' + r.note if r.note else ''}"
             for r in retrieved.itertuples()]
    return ("You categorize a college student's bank charges. Use ONLY the evidence below. "
            "Cite chunk IDs in [brackets] for every claim. If the evidence is not enough, reply "
            "'ASK STUDENT'.\n\nEVIDENCE:\n" + "\n".join(lines) +
            f"\n\nNEW CHARGE: {description} | ${abs(amount):.2f}\nCategory and reason:")

## Step 5. The A/B test

| | Condition A (baseline) | Condition B |
|---|---|---|
| **top-k** | **1** | **5** |
| Sources, chunks, cleaning, similarity method, answer rule | same | same |

**Hypothesis:** with k = 1 the system just copies the nearest charge, so at mixed stores it's a coin flip. With k = 5 it can *see* that a store has mixed labels and use the amount to decide. The risk: more chunks also means more chances to pull in misleading evidence.

### The five test cases (all real September charges, never seen by the book)

| # | Type (from handout) | Charge | Our label | Why we picked it |
|---|---|---|---|---|
| 1 | Direct | Price Chopper $74.32 | Groceries | Store appears in history many times. Easy case. |
| 2 | Paraphrase | Hudson Valley Urgent Care $35.00 | Health | Never seen before; evidence only exists in *different words* (the guide keyword). Also shares "HUDSON" with our paycheck. |
| 3 | Multi-part / across chunks | Target $61.44 | Shopping | Mixed store; answering needs several chunks (food runs **and** dorm-stuff trips) plus amounts. |
| 4 | Distractor / conflicting | CVS $13.93 | Health (cold medicine) | History disagrees: CVS was Health once, Shopping once. |
| 5 | Out-of-scope / absent | PayPal *KJ8STUDIO $18.00 | Shopping (phone case, unknown seller) | No evidence about this seller exists. Correct behavior = ask the student. |

`expected` = the chunk IDs that would actually justify the correct answer. Case 5 has none on purpose.

In [ ]:
CASES = [
    # (case, test charge id, chunk IDs that would justify the correct answer)
    ("1 Direct", 156, [c for c in s1.loc[s1["text"].str.contains("PRICE CHOPPER"), "chunk_id"]] + ["G-GROC-00"]),
    ("2 Paraphrase", 199, ["G-HEALTH-04"]),
    ("3 Multi-part", 195, list(s1.loc[s1["text"].str.contains("TARGET") & (s1["category"] == "Shopping"), "chunk_id"])),
    ("4 Distractor", 185, list(s1.loc[s1["text"].str.contains("CVS") & (s1["category"] == "Health"), "chunk_id"])),
    ("5 Out-of-scope", 181, []),
]
CONDITIONS = {"A": 1, "B": 5}

results = []
for case, tid, expected in CASES:
    t = test.loc[test["id"] == tid].iloc[0]
    print("=" * 110)
    print(f"CASE {case}:  {t['description']}   ${abs(t['amount']):.2f}   (our label: {t['category']}; "
          f"note: {t['note'] if isinstance(t['note'], str) else '-'})")
    print(f"   expected evidence: {expected if expected else 'none exists'}")
    for cond, k in CONDITIONS.items():
        r = retrieve(t["description"], k)
        cat, cited, sentence = answer(r, t["amount"])
        print(f"\n  Condition {cond} (top-k = {k}) retrieved:")
        print(r[["chunk_id", "similarity", "text", "amount", "category", "note"]].to_string(index=False))
        print(f"  -> ANSWER: {sentence}  cites {cited}")
        results.append({"case": case, "cond": cond, "truth": t["category"], "answer": cat,
                        "correct": cat == t["category"], "cited": cited,
                        "expected_in_context": bool(set(expected) & set(r["chunk_id"])),
                        "citations_ok": bool(cited) and set(cited) <= set(expected),
                        "retrieved": list(r["chunk_id"])})

CASE 1 Direct:  PRICE CHOPPER #172 TROY NY   $74.32   (our label: Groceries; note: -)
   expected evidence: ['H-009', 'H-062', 'H-105', 'H-120', 'G-GROC-00']

  Condition A (top-k = 1) retrieved:
chunk_id  similarity                  text  amount  category note
   H-009         1.0 PRICE CHOPPER TROY NY  -49.45 Groceries     
  -> ANSWER: Groceries: matching past charges / guide entries say Groceries.  cites ['H-009']

  Condition B (top-k = 5) retrieved:
 chunk_id  similarity                  text  amount  category                   note
    H-009        1.00 PRICE CHOPPER TROY NY  -49.45 Groceries                       
    H-062        1.00 PRICE CHOPPER TROY NY  -48.23 Groceries                       
    H-105        1.00 PRICE CHOPPER TROY NY  -24.03 Groceries                       
    H-120        1.00 PRICE CHOPPER TROY NY  -29.48 Groceries                       
G-GROC-00        0.95         PRICE CHOPPER     NaN Groceries category guide keyword
  -> ANSWER: Groceries: matchi

### What to look at above

- **Case 1 (direct):** A and B both answer Groceries. B just retrieves more Price Chopper chunks that all agree. RAG works when the evidence is simple.
- **Case 2 (paraphrase):** the guide chunk `G-HEALTH-04` (`URGENT CARE`) scores 0.77, so both find it. In B, the other 4 chunks are our **paychecks** (`HUDSON DINING ... PAYROLL`, Income) — pulled in only because of the letters "HUDSON". They score 0.24, so rule 1 throws them out. Without that 0.50 cutoff, B would be looking at four "Income" chunks.
- **Case 3 (multi-part):** all four Target chunks tie at 1.00. **A** grabs the first one stored (`H-011`, a $20 food run) and says **Groceries — wrong.** **B** sees all four, notices the store is mixed, and picks the $63.13 trip (`H-049`, "dorm stuff") as closest to $61.44 → **Shopping, correct**, with a citation that actually justifies it.
- **Case 4 (distractor/conflicting):** CVS was Health ($22.10 allergy meds) once and Shopping ($9.48 toiletries) once. **A** happens to grab `H-037` → Health (right, but by luck of storage order). **B** uses the amount rule: $13.93 is closer to $9.48 → **Shopping, wrong.** The amount isn't what decides a CVS category; *what was bought* is, and the bank text doesn't say. B made it **worse.**
- **Case 5 (out-of-scope):** cleaning deleted `KJ8STUDIO` (it has a digit in it), so the query became just `PAYPAL *`. That matches an old **eBay** purchase through PayPal (`H-079`, 0.59, just above the 0.50 cutoff). Both A and B answer Shopping and cite the eBay charge. The category happens to be right, but the **citation doesn't support it** — an eBay purchase says nothing about KJ8STUDIO. Correct behavior was "ask the student". This is the "looks credible but isn't" failure the handout asks about.

In [ ]:
summary = pd.DataFrame(results)
table = summary.pivot(index="case", columns="cond", values=["answer", "cited"])
table.columns = [f"{c[1]}: {c[0]}" for c in table.columns]
table.insert(0, "our label", summary.groupby("case")["truth"].first())
table

,our label,A: answer,B: answer,A: cited,B: cited
case,,,,,
1 Direct,Groceries,Groceries,Groceries,[H-009],"[H-009, H-062, H-105, H-120, G-GROC-00]"
2 Paraphrase,Health,Health,Health,[G-HEALTH-04],[G-HEALTH-04]
3 Multi-part,Shopping,Groceries,Shopping,[H-011],[H-049]
4 Distractor,Health,Health,Shopping,[H-037],[H-091]
5 Out-of-scope,Shopping,Shopping,Shopping,[H-079],[H-079]


## Step 6. Check the citations (don't trust that a citation exists — read it)

The handout's evidence rule: *"The answer claims ___; chunk ___ says ___; therefore the claim is supported / unsupported."* This cell prints that sentence for every answer, using the text of the chunk that was cited. **"Supported" here means the cited chunk is one of the expected-evidence chunks** — i.e., it really is about this store and this kind of purchase.

In [ ]:
by_id = chunks.set_index("chunk_id")
for row in results:
    if not row["cited"]:
        print(f"Case {row['case']} [{row['cond']}]: answer = ask the student (no claim made)")
        continue
    c = by_id.loc[row["cited"][0]]
    amt = "" if pd.isna(c["amount"]) else f" ${abs(c['amount']):.2f}"
    verdict = "SUPPORTED" if row["citations_ok"] else "UNSUPPORTED"
    print(f"Case {row['case']} [{row['cond']}]: the answer claims {row['answer']}; chunk {row['cited'][0]} says "
          f"'{c['text']}{amt}' = {c['category']}{' (' + c['note'] + ')' if c['note'] else ''}; -> {verdict}")

print()
for cond in CONDITIONS:
    s = summary[summary["cond"] == cond]
    print(f"Condition {cond}: relevant evidence reached context {s['expected_in_context'].sum()}/4 answerable cases | "
          f"correct answers {s['correct'].sum()}/5 | citations point to supporting chunks {s['citations_ok'].sum()}/5 | "
          f"safe on out-of-scope case: {'yes' if s.iloc[-1]['answer'] == 'ASK STUDENT' else 'no'}")

Case 1 Direct [A]: the answer claims Groceries; chunk H-009 says 'PRICE CHOPPER TROY NY $49.45' = Groceries; -> SUPPORTED
Case 1 Direct [B]: the answer claims Groceries; chunk H-009 says 'PRICE CHOPPER TROY NY $49.45' = Groceries; -> SUPPORTED
Case 2 Paraphrase [A]: the answer claims Health; chunk G-HEALTH-04 says 'URGENT CARE' = Health (category guide keyword); -> SUPPORTED
Case 2 Paraphrase [B]: the answer claims Health; chunk G-HEALTH-04 says 'URGENT CARE' = Health (category guide keyword); -> SUPPORTED
Case 3 Multi-part [A]: the answer claims Groceries; chunk H-011 says 'TARGET LATHAM NY $20.27' = Groceries (food run); -> UNSUPPORTED
Case 3 Multi-part [B]: the answer claims Shopping; chunk H-049 says 'TARGET LATHAM NY $63.13' = Shopping (apartment / dorm stuff); -> SUPPORTED
Case 4 Distractor [A]: the answer claims Health; chunk H-037 says 'CVS/PHARMACY TROY NY $22.10' = Health (allergy meds); -> SUPPORTED
Case 4 Distractor [B]: the answer claims Shopping; chunk H-091 says 'CVS/PHA

## Step 7. Same A/B on all 68 September charges

Five cases tell a story; the full month tells us if the story holds. Same pipeline, every September charge.

In [ ]:
rows = []
for _, t in test.iterrows():
    row = {"id": t["id"], "charge": t["clean"], "amount": t["amount"], "our label": t["category"]}
    for cond, k in CONDITIONS.items():
        row[cond], _, _ = answer(retrieve(t["description"], k), t["amount"])
    rows.append(row)
full = pd.DataFrame(rows)

for cond in CONDITIONS:
    right = (full[cond] == full["our label"]).sum()
    asked = (full[cond] == "ASK STUDENT").sum()
    print(f"Condition {cond}: right {right}/68 ({right / 68:.0%}), asked the student {asked}, "
          f"wrong {68 - right - asked}")

print("\nCharges where A and B disagree, or either is wrong:")
full[(full["A"] != full["B"]) | (full["A"] != full["our label"]) | (full["B"] != full["our label"])]

Condition A: right 64/68 (94%), asked the student 0, wrong 4
Condition B: right 65/68 (96%), asked the student 1, wrong 2

Charges where A and B disagree, or either is wrong:


,id,charge,amount,our label,A,B
23,167,APPLE.COM/BILL CUPERTINO CA,-0.99,Entertainment,Subscriptions,ASK STUDENT
41,185,CVS/PHARMACY TROY NY,-13.93,Health,Health,Shopping
51,195,TARGET LATHAM NY,-61.44,Shopping,Groceries,Shopping
57,201,STEWARTS SHOP TROY NY,-5.53,Dining,Transport,Dining
65,209,COSTCO WHSE COLONIE NY,-64.99,Shopping,Groceries,Groceries


B gets **2 more right than A on mixed stores** (Target $61.44, Stewart's $5.53), turns one confident wrong answer into "ask the student" (Apple $0.99 — not close in amount to anything), and **loses one** (CVS). Costco ($64.99 desk chair) is wrong in both: all the Costco evidence (one bulk-food trip, plus the guide keyword `COSTCO` → Groceries) says Groceries, so the answer is backed by its citations and still wrong. That's the handout's **E1** idea: an answer can be well-cited and wrong if the evidence itself doesn't cover the situation.

## Step 8 (extra, not part of the A/B). Does chunk size matter for the guide?

We cut the guide into one chunk **per keyword**. What if we'd used one chunk **per category** (all Health keywords in one chunk)? Only the guide changes here; k = 5 and everything else stay the same.

In [ ]:
s2_big = pd.DataFrame([
    {"chunk_id": f"G-{short}", "source": "S2 guide", "text": " ".join(kws), "date": "", "amount": np.nan,
     "category": cat, "note": "category guide"} for cat, short, kws in GUIDE
])
for desc in ["HUDSON VALLEY URGENT CARE TROY NY", "TM *TICKETMASTER EVENTS", "POS PURCHASE UNIQLO USA LLC NEW YORK NY"]:
    for name, guide_chunks in [("small chunks (1 keyword)", s2), ("big chunks (1 category)", s2_big)]:
        pool = pd.concat([s1, guide_chunks], ignore_index=True)
        v = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True).fit(pool["text"])
        sc = (v.transform([normalize(desc)]) @ v.transform(pool["text"]).T).toarray()[0]
        best = int(np.argmax(sc))
        print(f"{desc[:34]:36s} {name:26s} best chunk {pool.loc[best, 'chunk_id']:12s} "
              f"sim {sc[best]:.2f} -> {pool.loc[best, 'category']}")

HUDSON VALLEY URGENT CARE TROY NY    small chunks (1 keyword)   best chunk G-HEALTH-04  sim 0.77 -> Health
HUDSON VALLEY URGENT CARE TROY NY    big chunks (1 category)    best chunk G-HEALTH     sim 0.33 -> Health
TM *TICKETMASTER EVENTS              small chunks (1 keyword)   best chunk G-ENT-03     sim 0.96 -> Entertainment
TM *TICKETMASTER EVENTS              big chunks (1 category)    best chunk G-ENT        sim 0.41 -> Entertainment
POS PURCHASE UNIQLO USA LLC NEW YO   small chunks (1 keyword)   best chunk G-SHOP-10    sim 0.68 -> Shopping
POS PURCHASE UNIQLO USA LLC NEW YO   big chunks (1 category)    best chunk H-103        sim 0.38 -> Transport


With big chunks, the one keyword that matters (`URGENT CARE`) is buried among 7 others, so the similarity score drops below our 0.50 cutoff (or a wrong chunk like `MTA ... NEW YORK NY` wins for Uniqlo). The evidence is *in* the source but retrieval misses it — handout failure code **R2 (chunk boundary)** / **R1 (retrieval miss)**. Small chunks work better for short bank text.

## Step 9 (optional). Try a real LLM as the answer step

The cell below prints the exact prompt for case 3 under condition B. Paste it into Claude (no tools, fresh chat) and compare its answer and citations to our rule's answer. If you do this, note in the worksheet that generation was done by an LLM for that case.

In [ ]:
t = test.loc[test["id"] == 195].iloc[0]
print(build_prompt(t["description"], t["amount"], retrieve(t["description"], 5)))

You categorize a college student's bank charges. Use ONLY the evidence below. Cite chunk IDs in [brackets] for every claim. If the evidence is not enough, reply 'ASK STUDENT'.

EVIDENCE:
[H-011] TARGET LATHAM NY | $20.27 | category: Groceries | note: food run
[H-049] TARGET LATHAM NY | $63.13 | category: Shopping | note: apartment / dorm stuff
[H-075] TARGET LATHAM NY | $22.51 | category: Groceries | note: food run
[H-126] TARGET LATHAM NY | $63.40 | category: Shopping | note: apartment / dorm stuff
[G-SHOP-02] TARGET | category: Shopping | note: category guide keyword

NEW CHARGE: POS PURCHASE TARGET T-1849 LATHAM NY | $61.44
Category and reason:


## Summary for the worksheet

- **A/B effect:** going from top-k = 1 to top-k = 5 fixed mixed stores where the amount separates the purchase types (Target, Stewart's) and made one wrong answer an "ask the student", but made CVS worse, because there the amount doesn't separate anything.
- **Biggest risk:** a weak lookalike chunk just above the cutoff (PayPal/eBay, 0.59) produces a confident, cited, *unsupported* answer when the evidence is actually absent.
- **Compared to A2:** the LLM alone agreed with our labels on 93–94% of these same 68 charges. RAG with k = 5 gets 96% (65/68) and every answer comes with a chunk ID we can check. That is not a fair race, though: RAG gets to read the student's own labeled history and the LLM did not.

The filled-in worksheet is `RAG_Handout.docx` in this folder.